In [ ]:
import os

from sympy.physics.units import temperature

from Chatbot.Structured.pydanticoutputparser import prompt

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_API_KEY"


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import  FAISS
from langchain_core.prompts import  PromptTemplate

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
video_id = '-HzgcbRXUK8'

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])

    transcript = " ".join(chunk.text for chunk in fetched_transcript)
    print(transcript)

except TranscriptsDisabled:
    print('No Caption available for this video')

In [ ]:
print(fethed_transcript)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
vectore_store = FAISS.from_documents(chunks,embeddings)

In [ ]:
vectore_store.index_to_docstore_id

In [ ]:
vectore_store.get_by_ids(['3600104a-bb29-4300-a604-40f1d73017ae'])

In [ ]:
retriver = vectore_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [ ]:
retriver

In [ ]:
retriver.invoke('What is AI')

In [ ]:
llm = ChatGoogleGenerativeAI(model='gemini-3-flash-preview', api_key=GEMINI_API_KEY,temperature=0.2)

In [ ]:
prompt = PromptTemplate(template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,input_variables=['context','question'])

In [ ]:
question = 'Is the topic of aliens discussed in this video?If yes then what  was discussed?'
retriver_docs = retriver.invoke(question)

In [ ]:
retriver_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retriver_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({'context':context_text,'question':question})
final_prompt

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.text)

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_doc(retriver_doc):
    context_text = "\n\n".join(doc.page_content for doc in retriver_doc)
    return  context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context':retriver | RunnableLambda(format_doc),
    'question':RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('What is AI')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt| llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video?')